<a href="https://colab.research.google.com/github/gannonmalex/STOX/blob/main/noobai_lora_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# NoobAI-XL 1.1 — LoRA Training (DreamBooth SDXL Advanced)

**Runtime → Change runtime type → A100 GPU** (or L4/T4 if budget is tight)

This notebook trains a LoRA on [Laxhar/noobai-XL-1.1](https://huggingface.co/Laxhar/noobai-XL-1.1) using per-image captions.

## 1. Install dependencies

In [1]:
!pip install -q git+https://github.com/huggingface/diffusers.git
!pip install -q accelerate transformers peft datasets safetensors ftfy tensorboard sentencepiece bitsandbytes

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 46.0 MB/s eta 0:00:00


## 2. Upload your dataset

Upload a **zip** called `noobai_character.zip` containing:
```
noobai_character/
  0000.png
  0000.txt
  0001.png
  0001.txt
  ...
```
Each `.txt` has the caption/tags for the matching `.png`.

**Option A** — Upload via the Colab file browser (left sidebar → 📁 → upload `noobai_character.zip`)  
**Option B** — Upload from Google Drive (uncomment the Drive cell below)

In [2]:
# === Option A: upload zip directly ===
# After running this cell, click "Choose Files" and pick noobai_character.zip
from google.colab import files
uploaded = files.upload()  # upload noobai_character.zip

Saving noobai_character.zip to noobai_character.zip


In [3]:
# === Option B: load from Google Drive (uncomment to use) ===
# from google.colab import drive
# drive.mount('/content/drive')
# !cp /content/drive/MyDrive/noobai_character.zip /content/

In [4]:
!unzip -qo noobai_character.zip -d /content/dataset_raw
!ls /content/dataset_raw/noobai_character/ | head -20

0000.png
0000.txt
0001.png
0001.txt
0002.png
0002.txt
0003.png
0003.txt
0004.png
0004.txt
0005.png
0005.txt
0006.png
0006.txt
0007.png
0007.txt
0009.png
0009.txt
0010.png
0010.txt


## 3. Build `metadata.jsonl` from your `.txt` caption files

This creates the format the training script expects when using `--dataset_name`.

In [5]:
import json, os, shutil
from pathlib import Path

RAW_DIR = Path("/content/dataset_raw/noobai_character")
DATASET_DIR = Path("/content/dataset")
DATASET_DIR.mkdir(exist_ok=True)

# Pair up images and captions, write metadata.jsonl
entries = []
for img_path in sorted(RAW_DIR.glob("*.png")):
    txt_path = img_path.with_suffix(".txt")
    # Use per-image caption if available, otherwise fall back to a default
    if txt_path.exists():
        caption = txt_path.read_text().strip()
    else:
        caption = "masterpiece, best quality, VULGAR, 1girl"
    # Copy image to clean dataset folder
    dest = DATASET_DIR / img_path.name
    shutil.copy2(img_path, dest)
    entries.append({"file_name": img_path.name, "text": caption})

# Also handle .jpg / .jpeg / .webp if you have any
for ext in ("*.jpg", "*.jpeg", "*.webp"):
    for img_path in sorted(RAW_DIR.glob(ext)):
        txt_path = img_path.with_suffix(".txt")
        caption = txt_path.read_text().strip() if txt_path.exists() else "masterpiece, best quality, VULGAR, 1girl"
        dest = DATASET_DIR / img_path.name
        shutil.copy2(img_path, dest)
        entries.append({"file_name": img_path.name, "text": caption})

with open(DATASET_DIR / "metadata.jsonl", "w") as f:
    for e in entries:
        f.write(json.dumps(e) + "\n")

print(f"✓ {len(entries)} images with captions")
print("First 3 entries:")
for e in entries[:3]:
    print(f"  {e['file_name']}: {e['text'][:80]}...")

✓ 28 images with captions
First 3 entries:
  0000.png: VULGAR, my_style, 1girl, close-up, smoking, cigarette, blue lips, full lips, glo...
  0001.png: VULGAR, my_style, 1girl, lying down, fishnets, holding cigarette, smoking, marti...
  0002.png: VULGAR, my_style, 1girl, asian, blue lips, full lips, glossy lips, smoking, ciga...


## 4. Check GPU

In [6]:
!nvidia-smi

Thu Mar 12 09:40:49 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   31C    P0             51W /  400W |       0MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

## 5. Clone diffusers (for the training script)

In [7]:
!git clone --depth 1 https://github.com/huggingface/diffusers.git /content/diffusers

fatal: destination path '/content/diffusers' already exists and is not an empty directory.


## 6. Login to Hugging Face (needed to download the model)

Get a token from https://huggingface.co/settings/tokens

In [8]:
from huggingface_hub import notebook_login
notebook_login()

## 7. Launch training

### Key parameters you may want to tweak:
| Param | Default | Notes |
|-------|---------|-------|
| `--max_train_steps` | 2000 | More steps = longer but potentially better |
| `--learning_rate` | 1e-4 | Try 5e-5 if you see overfitting |
| `--rank` | 16 | LoRA rank; 8-32 typical |
| `--resolution` | 1024 | Native SDXL res |
| `--train_batch_size` | 1 | Increase to 2 on A100 80GB |
| `--gradient_accumulation_steps` | 4 | Effective batch = batch_size × this |
| `--mixed_precision` | fp16 | Use bf16 on A100 |
| `--checkpointing_steps` | 500 | Save checkpoints every N steps |

In [1]:
MODEL_NAME   = "Laxhar/noobai-XL-1.1"         # HF model ID
DATASET_DIR  = "/content/dataset"              # folder with images + metadata.jsonl
OUTPUT_DIR   = "/content/noobai_VULGAR_lora"   # where LoRA weights are saved

RESOLUTION   = 1024
BATCH_SIZE   = 1      # bump to 2 on A100-80GB
GRAD_ACCUM   = 4      # effective batch = BATCH_SIZE * GRAD_ACCUM
LR           = 1e-4
MAX_STEPS    = 3000
RANK         = 16
MIXED_PREC   = "fp16" # use "bf16" on A100, "fp16" on T4/L4
CKPT_STEPS   = 100    # save a checkpoint every N steps
VALIDATION_STEPS = 150 # output example images every N steps

In [2]:
!accelerate launch /content/diffusers/examples/text_to_image/train_text_to_image_lora_sdxl.py \
  --pretrained_model_name_or_path="{MODEL_NAME}" \
  --train_data_dir="{DATASET_DIR}" \
  --caption_column="text" \
  --resolution={RESOLUTION} \
  --train_batch_size={BATCH_SIZE} \
  --gradient_accumulation_steps={GRAD_ACCUM} \
  --learning_rate={LR} \
  --lr_scheduler="cosine" \
  --lr_warmup_steps=100 \
  --max_train_steps={MAX_STEPS} \
  --mixed_precision="{MIXED_PREC}" \
  --rank={RANK} \
  --checkpointing_steps={CKPT_STEPS} \
  --validation_prompt="masterpiece, best quality, VULGAR, 1girl" \
  --validation_steps={VALIDATION_STEPS} \
  --seed=42 \
  --output_dir="{OUTPUT_DIR}"

The following values were not passed to `accelerate launch` and had defaults used instead:
	`--num_processes` was set to a value of `1`
	`--num_machines` was set to a value of `1`
	`--mixed_precision` was set to a value of `'no'`
	`--dynamo_backend` was set to a value of `'no'`
To avoid this warning pass in values for each of the problematic parameters or run `accelerate config`.
/usr/bin/python3: can't open file '/content/diffusers/examples/text_to_image/train_text_to_image_lora_sdxl.py': [Errno 2] No such file or directory
Traceback (most recent call last):
  File "/usr/local/bin/accelerate", line 10, in <module>
    sys.exit(main())
             ^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/accelerate/commands/accelerate_cli.py", line 50, in main
    args.func(args)
  File "/usr/local/lib/python3.12/dist-packages/accelerate/commands/launch.py", line 1405, in launch_command
    simple_launcher(args)
  File "/usr/local/lib/python3.12/dist-packages/accelerate/commands/launch.p

## 8. Test the trained LoRA

In [15]:
import torch
from diffusers import StableDiffusionXLPipeline, DPMSolverMultistepScheduler

pipe = StableDiffusionXLPipeline.from_pretrained(
    MODEL_NAME, torch_dtype=torch.float16
).to("cuda")

pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config)

# Load LoRA weights from local directory
pipe.load_lora_weights(OUTPUT_DIR, weight_name="pytorch_lora_weights.safetensors")

prompt = "masterpiece, best quality, VULGAR, 1girl, solo, standing, looking at viewer"
negative = "worst quality, low quality, blurry, deformed"

image = pipe(
    prompt,
    negative_prompt=negative,
    num_inference_steps=28,
    guidance_scale=7.0,
    width=832,
    height=1216,
).images[0]

image

Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/517 [00:00<?, ?it/s]

OSError: We couldn't connect to 'https://huggingface.co' to load this model, couldn't find it in the cached files and it looks like /content/noobai_VULGAR_lora is not the path to a directory containing a file named pytorch_lora_weights.safetensors or 
Checkout your internet connection or see how to run the library in offline mode at 'https://huggingface.co/docs/diffusers/installation#offline-mode'.

In [6]:
!find /content/diffusers -name train_text_to_image_lora_sdxl.py

/content/diffusers/examples/research_projects/scheduled_huber_loss_training/text_to_image/train_text_to_image_lora_sdxl.py
/content/diffusers/examples/text_to_image/train_text_to_image_lora_sdxl.py


In [5]:
# Re-clone diffusers to ensure the training script exists
!rm -rf /content/diffusers
!git clone --depth 1 https://github.com/huggingface/diffusers.git /content/diffusers

Cloning into '/content/diffusers'...
remote: Enumerating objects: 2733, done.
remote: Counting objects: 100% (2733/2733), done.
remote: Compressing objects: 100% (1891/1891), done.
remote: Total 2733 (delta 1133), reused 1227 (delta 814), pack-reused 0 (from 0)
Receiving objects: 100% (2733/2733), 8.71 MiB | 17.67 MiB/s, done.
Resolving deltas: 100% (1133/1133), done.


In [4]:
!find /content/diffusers -name train_text_to_image_lora_sdxl.py

find: ‘/content/diffusers’: No such file or directory


## 9. Download the LoRA weights

In [11]:
# Zip and download
!cd /content && zip -r noobai_VULGAR_lora.zip noobai_VULGAR_lora/

from google.colab import files
files.download("/content/noobai_VULGAR_lora.zip")

	zip warning: name not matched: noobai_VULGAR_lora/

zip error: Nothing to do! (try: zip -r noobai_VULGAR_lora.zip . -i noobai_VULGAR_lora/)


FileNotFoundError: Cannot find file: /content/noobai_VULGAR_lora.zip

In [ ]:
# === OR save to Google Drive ===
# from google.colab import drive
# drive.mount('/content/drive')
# !cp -r /content/noobai_VULGAR_lora /content/drive/MyDrive/